# Тестирование ResNet модели детекции углов шахматной доски

Этот блокнот для тестирования обученной ResNet модели на ваших изображениях.

**Инструкция:**
1. Поместите ваше тестовое изображение в папку `chess-recognition/` (или укажите полный путь)
2. Укажите путь к изображению в ячейке ниже (перетащите файл или введите путь)
3. Запустите все ячейки

## 1. Импорты и настройки

In [11]:
import sys
from pathlib import Path
import cv2
import numpy as np
from PIL import Image

# Добавляем путь к модулям
sys.path.insert(0, str(Path.cwd() / 'src'))

# Импортируем функцию детекции углов
from improved_board_mapping import _detect_board_corners_resnet, CornerRegressor

# Проверка доступности matplotlib
try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as e:
    MATPLOTLIB_AVAILABLE = False
    print(f"Matplotlib недоступен ({e}), будет использован OpenCV для отображения")

print("Импорты успешны!")

Импорты успешны!


## 2. Укажите путь к изображению

**Перетащите файл изображения в эту ячейку или укажите путь к файлу:**

In [12]:
# УКАЖИТЕ ПУТЬ К ВАШЕМУ ИЗОБРАЖЕНИЮ ЗДЕСЬ
# Например: "image.jpg" (если файл в той же папке)
# Или: "D:/chesscast/chess-recognition/test.jpg" (полный путь)
IMAGE_PATH = "image.jpg"  # <-- ИЗМЕНИТЕ НА ВАШ ПУТЬ

# Проверка существования файла
img_path = Path(IMAGE_PATH)
if not img_path.exists():
    print(f"⚠️ Файл не найден: {IMAGE_PATH}")
    print(f"Текущая рабочая директория: {Path.cwd()}")
    print("\nДоступные файлы в текущей директории:")
    for f in Path.cwd().glob("*.jpg"):
        print(f"  - {f.name}")
    for f in Path.cwd().glob("*.png"):
        print(f"  - {f.name}")
else:
    print(f"✅ Файл найден: {img_path.absolute()}")

✅ Файл найден: d:\chesscast\chess-recognition\image.jpg


## 3. Загрузка изображения

In [13]:
# Загрузка изображения через OpenCV (BGR формат)
image = cv2.imread(str(img_path))
if image is None:
    raise FileNotFoundError(f"Не удалось загрузить изображение: {IMAGE_PATH}")

# Конвертация BGR -> RGB для отображения
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

h, w = image.shape[:2]
print(f"✅ Изображение загружено: {w}x{h} пикселей")

# Показываем оригинальное изображение
if MATPLOTLIB_AVAILABLE:
    try:
        plt.figure(figsize=(12, 8))
        plt.imshow(image_rgb)
        plt.title(f"Оригинальное изображение ({w}x{h})")
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Ошибка matplotlib: {e}")
        print("Используем OpenCV для отображения")
        cv2.imshow("Оригинальное изображение", image)
        print("Нажмите любую клавишу для продолжения...")
        cv2.waitKey(0)
        cv2.destroyAllWindows()
else:
    cv2.imshow("Оригинальное изображение", image)
    print("Нажмите любую клавишу для продолжения...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

✅ Изображение загружено: 1200x1600 пикселей
Ошибка matplotlib: 'RcParams' object has no attribute '_get'
Используем OpenCV для отображения
Нажмите любую клавишу для продолжения...


## 4. Детекция углов доски через ResNet модель

In [14]:
# Запуск детекции углов
# Модель будет загружена автоматически (путь по умолчанию)
print("🔍 Запуск детекции углов доски...")
print("=" * 60)

# Определяем устройство (GPU если доступно)
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используемое устройство: {device}")

# Вызываем функцию детекции
corners = _detect_board_corners_resnet(image, model_path=None, img_size=640, device=device)

if corners is not None:
    print(f"\n✅ Углы доски успешно обнаружены!")
    print(f"Найдено углов: {len(corners)}")
    print("\nКоординаты углов (x, y):")
    corner_names = ["TL (верх-лево)", "TR (верх-право)", "BR (низ-право)", "BL (низ-лево)"]
    for i, (x, y) in enumerate(corners):
        print(f"  {corner_names[i]}: ({x:.1f}, {y:.1f})")
else:
    print("\n❌ Не удалось обнаружить углы доски")
    print("Возможные причины:")
    print("  - Модель не найдена (проверьте путь к модели)")
    print("  - Изображение не содержит шахматную доску")
    print("  - Ошибка при обработке изображения")

[RESNET] Путь к модели углов: D:\chesscast\chess-recognition\best_resnet34_board_corners.pt


🔍 Запуск детекции углов доски...
Используемое устройство: cuda


[RESNET] Нормализованные координаты от модели: [    0.16599     0.25789     0.85346     0.22783     0.92058     0.80421    0.095671     0.78959]
[RESNET] Размер исходного изображения: 1200x1600
[RESNET] Координаты в пикселях (до сортировки): [[199.19375610351562, 412.63104248046875], [1024.154541015625, 364.52020263671875], [1104.6961669921875, 1286.7337646484375], [114.805419921875, 1263.3470458984375]]
[RESNET] Детекция успешна, углы: [[199.19375610351562, 412.63104248046875], [1024.154541015625, 364.52020263671875], [1104.6961669921875, 1286.7337646484375], [114.805419921875, 1263.3470458984375]]



✅ Углы доски успешно обнаружены!
Найдено углов: 4

Координаты углов (x, y):
  TL (верх-лево): (199.2, 412.6)
  TR (верх-право): (1024.2, 364.5)
  BR (низ-право): (1104.7, 1286.7)
  BL (низ-лево): (114.8, 1263.3)


## 5. Визуализация результатов

In [15]:
if corners is not None:
    # Создаем копию изображения для визуализации
    vis_image = image_rgb.copy()
    
    # Цвета для углов (RGB)
    colors = [
        (255, 0, 0),    # Красный - TL
        (0, 255, 0),    # Зеленый - TR
        (0, 0, 255),    # Синий - BR
        (255, 255, 0)   # Желтый - BL
    ]
    
    corner_names = ["TL", "TR", "BR", "BL"]
    
    # Рисуем углы на изображении
    for i, (x, y) in enumerate(corners):
        color = colors[i]
        # Рисуем большой круг
        cv2.circle(vis_image, (int(x), int(y)), 20, color, -1)
        # Рисуем обводку
        cv2.circle(vis_image, (int(x), int(y)), 20, (255, 255, 255), 3)
        # Подписываем угол
        cv2.putText(vis_image, corner_names[i], (int(x) + 25, int(y) + 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
        cv2.putText(vis_image, corner_names[i], (int(x) + 25, int(y) + 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 2)
    
    # Рисуем линии между углами (контур доски)
    pts = corners.astype(int).reshape((-1, 1, 2))
    cv2.polylines(vis_image, [pts], True, (255, 255, 255), 4)
    cv2.polylines(vis_image, [pts], True, (0, 255, 255), 2)
    
    # Показываем результат
    if MATPLOTLIB_AVAILABLE:
        try:
            fig, axes = plt.subplots(1, 2, figsize=(20, 10))
            
            # Оригинал
            axes[0].imshow(image_rgb)
            axes[0].set_title("Оригинальное изображение", fontsize=14)
            axes[0].axis('off')
            
            # С результатами
            axes[1].imshow(vis_image)
            axes[1].set_title("Результаты детекции углов доски (ResNet)", fontsize=14)
            axes[1].axis('off')
            
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Ошибка matplotlib: {e}")
            print("Используем OpenCV для отображения")
            vis_image_bgr = cv2.cvtColor(vis_image, cv2.COLOR_RGB2BGR)
            cv2.imshow("Оригинальное изображение", image)
            cv2.imshow("Результаты детекции углов", vis_image_bgr)
            print("Нажмите любую клавишу для закрытия окон...")
            cv2.waitKey(0)
            cv2.destroyAllWindows()
    else:
        vis_image_bgr = cv2.cvtColor(vis_image, cv2.COLOR_RGB2BGR)
        cv2.imshow("Оригинальное изображение", image)
        cv2.imshow("Результаты детекции углов", vis_image_bgr)
        print("Нажмите любую клавишу для закрытия окон...")
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    
    # Легенда
    print("\n" + "=" * 60)
    print("ЛЕГЕНДА:")
    print("=" * 60)
    for i, name in enumerate(corner_names):
        color_name = ['Красный', 'Зеленый', 'Синий', 'Желтый'][i]
        print(f"  {name} ({corner_names[i]}) - {color_name} цвет")
else:
    print("Нечего визуализировать - углы не найдены")

Ошибка matplotlib: 'RcParams' object has no attribute '_get'
Используем OpenCV для отображения
Нажмите любую клавишу для закрытия окон...

ЛЕГЕНДА:
  TL (TL) - Красный цвет
  TR (TR) - Зеленый цвет
  BR (BR) - Синий цвет
  BL (BL) - Желтый цвет


## 6. Сохранение результата

In [ ]:
if corners is not None:
    # Сохраняем результат
    output_path = f"result_resnet_{Path(IMAGE_PATH).stem}.jpg"
    
    # Конвертируем RGB -> BGR для сохранения через OpenCV
    vis_image_bgr = cv2.cvtColor(vis_image, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_path, vis_image_bgr)
    
    print(f"✅ Результат сохранен: {output_path}")
    
    # Также сохраняем координаты в текстовый файл
    coords_path = f"corners_resnet_{Path(IMAGE_PATH).stem}.txt"
    with open(coords_path, 'w') as f:
        f.write("# Углы доски (x, y)\n")
        f.write("# Формат: TL, TR, BR, BL\n")
        for i, (x, y) in enumerate(corners):
            f.write(f"{x:.2f} {y:.2f}\n")
    print(f"✅ Координаты сохранены: {coords_path}")
else:
    print("Нечего сохранять - углы не найдены")